# Alfido Tech — Customer Transaction & Behavior Analysis
**Internship Project | Customer Segmentation, Purchase Patterns and Churn Risk**

Objective: analyze transactions and customer behavior, build RFM segments, visualize purchasing and retention patterns, and translate findings into actionable engagement recommendations.


## 1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("ecommerce_customer_data_custom_ratios.csv")
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"])

print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
display(df.head())
display(df.isna().sum())
print("Date range:", df["Purchase Date"].min(), "to", df["Purchase Date"].max())


## 2. Data cleaning decisions
- No duplicate rows were found.
- `Purchase Date` is converted to datetime.
- Core transaction fields are usable for RFM.
- `Returns` contains missing values. Missing return status is retained as unknown rather than automatically interpreted as no return.
- `Age` and `Customer Age` are redundant for the main RFM analysis.


In [ ]:
df["Returns_Unknown"] = df["Returns"].isna().astype(int)
df["Returns_Filled"] = df["Returns"].fillna(0).astype(int)


## 3. Customer-level feature engineering and RFM

In [ ]:
reference_date = df["Purchase Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    Recency=("Purchase Date", lambda x: (reference_date-x.max()).days),
    Frequency=("Purchase Date", "count"),
    Monetary=("Total Purchase Amount", "sum"),
    Avg_Order_Value=("Total Purchase Amount", "mean"),
    Total_Quantity=("Quantity", "sum"),
    First_Purchase=("Purchase Date", "min"),
    Last_Purchase=("Purchase Date", "max"),
    Churn=("Churn", "max")
).reset_index()

rfm["Tenure_Days"] = (rfm["Last_Purchase"]-rfm["First_Purchase"]).dt.days
rfm["R"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["RFM_Score"] = rfm[["R","F","M"]].sum(axis=1)


## 4. Segment customers

In [ ]:
def segment(row):
    R,F,M = row["R"],row["F"],row["M"]
    if R>=4 and F>=4 and M>=4: return "Champions"
    if R>=3 and F>=4: return "Loyal Customers"
    if R>=4 and F<=3: return "New / Promising"
    if R>=3 and F<=3: return "Potential Loyalists"
    if R<=2 and F>=3: return "At Risk"
    return "Lost / Hibernating"

rfm["Segment"] = rfm.apply(segment, axis=1)

segment_profile = rfm.groupby("Segment").agg(
    Customers=("Customer ID","size"),
    Avg_Recency=("Recency","mean"),
    Avg_Frequency=("Frequency","mean"),
    Avg_Spend=("Monetary","mean"),
    Churn_Rate=("Churn","mean")
).reset_index().sort_values("Customers", ascending=False)

display(segment_profile)


## 5. Segment visualization

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(segment_profile["Segment"], segment_profile["Customers"])
plt.title("Customer Distribution by RFM Segment")
plt.ylabel("Customers")
plt.xticks(rotation=30, ha="right")
plt.show()


## 6. Purchase patterns

In [ ]:
monthly = df.set_index("Purchase Date").resample("MS").agg(
    Transactions=("Customer ID","size"),
    Revenue=("Total Purchase Amount","sum"),
    Customers=("Customer ID","nunique")
).reset_index()

category = df.groupby("Product Category").agg(
    Transactions=("Customer ID","size"),
    Revenue=("Total Purchase Amount","sum"),
    Avg_Transaction=("Total Purchase Amount","mean")
).reset_index().sort_values("Revenue", ascending=False)

display(category)

plt.figure(figsize=(8,5))
plt.bar(category["Product Category"], category["Revenue"])
plt.title("Revenue by Product Category")
plt.ylabel("Total Purchase Amount")
plt.show()

plt.figure(figsize=(9,5))
plt.plot(monthly["Purchase Date"], monthly["Revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue")
plt.xlabel("Month")
plt.xticks(rotation=45, ha="right")
plt.show()


## 7. Retention and churn

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(monthly["Purchase Date"], monthly["Customers"], marker="o")
plt.title("Monthly Active Customer Trend")
plt.ylabel("Unique Customers")
plt.xlabel("Month")
plt.xticks(rotation=45, ha="right")
plt.show()

churn_by_segment = rfm.groupby("Segment")["Churn"].mean().reindex(segment_profile["Segment"])
plt.figure(figsize=(9,5))
plt.bar(churn_by_segment.index, churn_by_segment.values*100)
plt.title("Observed Churn Rate by RFM Segment")
plt.ylabel("Churn (%)")
plt.xticks(rotation=30, ha="right")
plt.show()

print("Overall observed churn:", round(df["Churn"].mean()*100,2), "%")


## 8. Findings
- Lost / Hibernating is the largest segment.
- Champions have the strongest combination of recency, frequency and spending.
- At-Risk customers combine relatively high historical value with long recency.
- Books and Clothing are the largest revenue contributors in this dataset.
- The supplied churn label is about 20% overall and differs only modestly across RFM segments; RFM should therefore be used for prioritization, not claimed as a standalone churn predictor.

## 9. Five actionable recommendations
1. Win back At-Risk customers using personalized, time-limited offers.
2. Protect Champions with loyalty benefits, early access and personalized cross-sell.
3. Test low-cost reactivation campaigns for Lost / Hibernating customers before increasing incentives.
4. Build a second-purchase journey for New / Promising customers.
5. Monitor churn by segment, category, recency, returns and monthly cohort in a recurring dashboard.
